# Week 6 — GrowClust3D.jl setup (trace1D mode)
Builds the four input files GrowClust3D.jl needs: velocity model (Z/Vp/Vs,
reused directly from the real Chow et al. 2022 profile already extracted for
NonLinLoc), event list, station list, and the control (.inp) file. Uses
`trace1D` mode (internal ray-tracing) rather than the NonLinLoc-grid modes,
since GrowClust3D.jl's grid reader only supports AZIMUTHAL_EQUIDIST, LAMBERT,
MERC and TRANS_MERC projections - not the TRANS SIMPLE convention our
NonLinLoc grids were built with (confirmed directly from GrowClust3D.jl's
own source, src/nllgrid.jl's check_proj function).

Cross-correlation data: our existing dt.cc is already in exactly the format
GrowClust3D.jl's xcordat_fmt=1 expects (confirmed against a real GrowClust3D
example file) - referenced directly, not duplicated.

In [ ]:
import os, re
import pandas as pd
import obspy

PROJECT_DIR = os.path.expanduser("~/tvz_project")
GC_DIR = os.path.join(PROJECT_DIR, "growclust")
GC_IN_DIR = os.path.join(GC_DIR, "in")
GC_OUT_DIR = os.path.join(GC_DIR, "out")
os.makedirs(GC_IN_DIR, exist_ok=True)
os.makedirs(GC_OUT_DIR, exist_ok=True)

INVENTORY_FILE = os.path.join(PROJECT_DIR, "data", "inventory", "NZ_2019-08-01_2019-09-30.xml")
CATALOG_DIR = os.path.join(PROJECT_DIR, "catalogs")
VEL2GRID_IN = os.path.join(PROJECT_DIR, "nonlinloc", "vel2grid_tvz.in")

TARGET_LAT, TARGET_LON = -38.8, 176.0  # same reference point used throughout

In [ ]:
# Reconstruct the velocity profile directly from our existing LAYER lines -
# no need to refetch the Chow et al. (2022) NetCDF data again.
rows = []
with open(VEL2GRID_IN) as f:
    for line in f:
        if line.startswith("LAYER"):
            parts = line.split()
            depth, vp, vs = float(parts[1]), float(parts[2]), float(parts[4])
            rows.append((depth, vp, vs))

vzmodel_path = os.path.join(GC_IN_DIR, "vzmodel.txt")
with open(vzmodel_path, "w") as f:
    for depth, vp, vs in rows:
        f.write(f"{depth:6.2f} {vp:6.3f} {vs:6.3f}\n")

print(f"Saved: {vzmodel_path} ({len(rows)} depth levels, {rows[0][0]} to {rows[-1][0]} km)")
print("First 3 rows:")
for r in rows[:3]:
    print(r)

In [ ]:
# Event list (evlist_fmt=1, "phase" format): yr mo dy hr mn sec lat lon depth mag eh ez rms evid
# evid MUST match dt.cc's event numbering exactly (event_idx + 1, same convention
# used when building dt.cc in the cross-correlation step).
filtered = pd.read_csv(os.path.join(CATALOG_DIR, "eqtransformer_nlloc_quality_filtered.csv"))
filtered["event_idx"] = filtered["file"].str.extract(r"_E(\d+)\.")[0].astype(int)
filtered["origin_dt"] = pd.to_datetime(filtered["origin_time"])

evlist_path = os.path.join(GC_IN_DIR, "evlist.txt")
with open(evlist_path, "w") as f:
    for _, row in filtered.iterrows():
        dt = row["origin_dt"]
        evid = int(row["event_idx"]) + 1
        f.write(
            f"{dt.year:4d} {dt.month:2d} {dt.day:2d} {dt.hour:2d} {dt.minute:2d} "
            f"{dt.second + dt.microsecond / 1e6:6.3f} "
            f"{row['lat']:9.5f} {row['lon']:10.5f} {row['depth_km']:7.3f} "
            f"0.00 {row['horiz_error_km']:6.3f} 0.000 {row['rms']:6.3f} {evid:8d}\n"
        )

print(f"Saved: {evlist_path} ({len(filtered)} events)")

In [ ]:
# Station list (stlist_fmt=2): bare station code (matching dt.cc's convention)
# lat lon elev_m. Filtered to only the 60 stations we actually have real data
# for (matching data/waveforms/), not the full ~209-station regional query -
# the extras have no picks, correlations, or travel-time grids behind them.
WAVEFORM_DIR = os.path.join(PROJECT_DIR, "data", "waveforms")
real_stations = {d.split(".")[1] for d in os.listdir(WAVEFORM_DIR) if d.startswith("NZ.")}
print(f"{len(real_stations)} stations with real downloaded data")

inventory = obspy.read_inventory(INVENTORY_FILE)

stlist_path = os.path.join(GC_IN_DIR, "stlist2.txt")
written = set()
with open(stlist_path, "w") as f:
    for net in inventory:
        for sta in net:
            if sta.code not in real_stations or sta.code in written:
                continue
            f.write(f"{sta.code:<6s} {sta.latitude:9.5f} {sta.longitude:10.5f} {sta.elevation:7.1f}\n")
            written.add(sta.code)

print(f"Saved: {stlist_path} ({len(written)} stations)")
missing = real_stations - written
if missing:
    print(f"WARNING: {len(missing)} real stations not found in inventory: {missing}")


In [ ]:
# Build the GrowClust3D.jl control file (trace1D mode).
# Projection: aeqd (Azimuthal Equidistant) - the closest available match to
# NonLinLoc's SIMPLE transform used elsewhere in the project, and appropriate
# since trace1D mode has no grid-header consistency check to satisfy (that
# check only applies to the nllgrid modes we moved away from).
os.makedirs(os.path.join(GC_OUT_DIR, "tt"), exist_ok=True)

dtcc_path = os.path.join(GC_DIR, "dt.cc")

inp_lines = [
    "****  TVZ EQTransformer/PyOcto/NonLinLoc Catalogue  *****",
    "******   GrowClust3D.jl relative relocation   *******",
    "*******************************************",
    "*",
    "*******************************************",
    "*************  Event list  ****************",
    "*******************************************",
    "* evlist_fmt (1 = phase, 2 = GrowClust, 3 = HypoInverse)",
    "1",
    "* fin_evlist (event list file name)",
    "in/evlist.txt",
    "*",
    "*******************************************",
    "************   Station list   *************",
    "*******************************************",
    "* stlist_fmt (1: station name, 2: incl. elev)",
    "2",
    "* fin_stlist (station list file name)",
    "in/stlist2.txt",
    "*",
    "*******************************************",
    "*************   XCOR data   ***************",
    "*******************************************",
    "* xcordat_fmt (1 = text), tdif_fmt (21 = tt2-tt1, 12 = tt1-tt2)",
    "1  12",
    "* fin_xcordat",
    dtcc_path,
    "*",
    "*******************************************",
    "*** Velocity Model / Travel Time Tables ***",
    "*******************************************",
    "* ttabsrc: travel time table source (\"trace\" or \"nllgrid\")",
    "trace",
    "* fin_vzmdl (model name)",
    "in/vzmodel.txt",
    "* fdir_ttab (directory for travel time tables/grids or NONE)",
    "out/tt/",
    "* projection (proj, ellps, lon0, lat0, rotANG, [latP1, latP2])",
    f"aeqd WGS84 {TARGET_LON} {TARGET_LAT} 0.0",
    "******************************************",
    "***** Travel Time Table Parameters  ******",
    "******************************************",
    "* vpvs_factor  rayparam_min",
    "  1.732             0.0",
    "* tt_zmin  tt_zmax  tt_zstep",
    "  -3.0        40.0       1.0",
    "* tt_xmin  tt_xmax  tt_xstep",
    "   0.0       200.0      1.0",
    "*",
    "******************************************",
    "***** GrowClust Algorithm Parameters *****",
    "******************************************",
    "* rmin  delmax rmsmax",
    "   0.6    80    0.2",
    "* rpsavgmin, rmincut  ngoodmin   iponly",
    "    0          0         0        0",
    "*",
    "******************************************",
    "************ Output files ****************",
    "******************************************",
    "* nboot  nbranch_min",
    "   0         2",
    "* fout_cat (relocated catalog)",
    "out/tvz.relocated.cat",
    "* fout_clust (relocated cluster file)",
    "out/tvz.relocated.clust",
    "* fout_log (program log)",
    "out/tvz.relocated.log",
    "* fout_boot (bootstrap distribution)",
    "NONE",
    "******************************************",
    "******************************************",
]

inp_path = os.path.join(GC_DIR, "growclust_tvz.inp")
with open(inp_path, "w") as f:
    f.write("\n".join(inp_lines) + "\n")

print(f"Saved: {inp_path}")
